In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ["NO_ALBUMENTATIONS_UPDATE"] = '1'
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [2]:
import cv2
import shutil 
import random
import numpy as np
import pandas as pd 
from glob import glob
import matplotlib.pyplot as plt
import albumentations as A

In [3]:
# def autocontrast(image, cutoff=3):
#     if len(image.shape) == 3:  # RGB image
#         channels = cv2.split(image)
#         auto_channels = [autocontrast_channel(ch, cutoff) for ch in channels]
#         return cv2.merge(auto_channels)
#     else:  # Grayscale image
#         return autocontrast_channel(image, cutoff)

# def autocontrast_channel(channel, cutoff):
#     # Calculate percentiles
#     low_val = np.percentile(channel, cutoff)
#     high_val = np.percentile(channel, 100 - cutoff)

#     # Stretch pixel values to [0, 255] range
#     if high_val > low_val:
#         scaled = (channel - low_val) * (255.0 / (high_val - low_val))
#         scaled = np.clip(scaled, 0, 255).astype(np.uint8)
#     else:
#         scaled = channel.copy()

#     return scaled

class AutoContrast(A.ImageOnlyTransform):
    def __init__(self, cutoff=3, always_apply=False, p=1.0):
        super().__init__(always_apply, p)
        self.cutoff = cutoff

    def autocontrast_channel(self, channel):
        low_val = np.percentile(channel, self.cutoff)
        high_val = np.percentile(channel, 100 - self.cutoff)

        if high_val > low_val:
            # Scale pixel values to [0, 255]
            scaled = (channel - low_val) * (255.0 / (high_val - low_val))
            return np.clip(scaled, 0, 255).astype(np.uint8)
        else:
            return channel.copy()

    def apply(self, img, **params):
        channels = cv2.split(img)
        auto_channels = [self.autocontrast_channel(ch).astype(np.uint8) for ch in channels]
        h, w = img.shape[:2]
        auto_channels = [cv2.resize(ch, (w, h)) for ch in auto_channels]
        return cv2.merge(auto_channels)

# ===================================

# class SaltAndPepperNoise(A.ImageOnlyTransform):
#     def __init__(self,salt_prob=0.01,pepper_prob=0.01,always_apply=True,p=1.0.0):
#         super(SaltAndPepperNoise, self).__init__(always_apply, p)
#         self.salt_prob = salt_prob
#         self.pepper_prob = pepper_prob

#     def apply(self,img,**params):
#         noisy_img = img.copy()
#         salt_mask = np.random.rand(*img.shape[:2]) < self.salt_prob
#         noisy_img[salt_mask] = 255
#         pepper_mask = np.random.rand(*img.shape[:2]) < self.pepper_prob
#         noisy_img[pepper_mask] = 0
#         return noisy_img

class SaltAndPepperNoise(A.ImageOnlyTransform):
    def __init__(self, salt_prob=0.01, pepper_prob=0.01, always_apply=False, p=1.0):
        super(SaltAndPepperNoise, self).__init__(always_apply, p)
        self.salt_prob = salt_prob
        self.pepper_prob = pepper_prob

    def apply(self,img,**params):
        noisy_img = img.copy()
        salt_mask = np.random.rand(*img.shape[:2]) < self.salt_prob
        noisy_img[salt_mask] = 255
        pepper_mask = np.random.rand(*img.shape[:2]) < self.pepper_prob
        noisy_img[pepper_mask] = 0
        return noisy_img

# ===================================

# class SpeckleNoise(A.ImageOnlyTransform):
#     def __init__(self,noise_prob=0.1,always_apply=True,p=1.0.0):
#         super(SpeckleNoise, self).__init__(always_apply,p)
#         self.noise_prob = noise_prob

#     def apply(self, img, **params):
#         noise = np.random.normal(0, self.noise_prob, img.shape)
#         noisy_img = img + img * noise
#         noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
#         return noisy_img

class SpeckleNoise(A.ImageOnlyTransform):
    def __init__(self, noise_prob=0.1, always_apply=False, p=1.0):
        super(SpeckleNoise, self).__init__(always_apply,p)
        self.noise_prob = noise_prob

    def apply(self, img, **params):
        noise = np.random.normal(0, self.noise_prob, img.shape)
        noisy_img = img + img * noise
        noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
        return noisy_img

# ===================================

# class EmbossEffect(A.ImageOnlyTransform):
#     def __init__(self, emboss_strength=1.0, always_apply=False, p=1.0.0):
#         super(EmbossEffect, self).__init__(always_apply, p)
#         self.emboss_strength = emboss_strength

#     def apply(self, img, **params):
#         kernel = np.array([[-2,-1,0],[-1,1,1],[0,1,2]],dtype=np.float32)
#         embossed_img = cv2.filter2D(img, -1, kernel * self.emboss_strength)
#         embossed_img = np.clip(embossed_img, 0, 255).astype(np.uint8)
#         return embossed_img

class EmbossEffect(A.ImageOnlyTransform):
    def __init__(self, emboss_strength=1.0, always_apply=False, p=1.0):
        super(EmbossEffect, self).__init__(always_apply, p)
        self.emboss_strength = emboss_strength

    def apply(self, img, **params):
        kernel = np.array([[-2,-1,0],[-1,1,1],[0,1,2]],dtype=np.float32)
        embossed_img = cv2.filter2D(img, -1, kernel * self.emboss_strength)
        embossed_img = np.clip(embossed_img, 0, 255).astype(np.uint8)
        return embossed_img

# ===================================

# def equalize_yuv(image):
#     yuv_image = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
#     yuv_image[:,:,0]=cv2.equalizeHist(yuv_image[:,:,0])
#     equalized_image = cv2.cvtColor(yuv_image, cv2.COLOR_YUV2BGR)
#     return equalized_image

class EqualizeYUV(A.ImageOnlyTransform):
    def __init__(self, always_apply=False, p=1.0):
        super().__init__(always_apply, p)

    def apply(self, image, **kwargs):
        yuv_image = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
        y_channel = yuv_image[:, :, 0].astype(np.uint8)
        yuv_image[:, :, 0] = cv2.equalizeHist(y_channel)
        return cv2.cvtColor(yuv_image, cv2.COLOR_YUV2BGR)

# ===================================

# def apply_vignette(input_image, **kwargs):  
#     rows, cols = input_image.shape[:2]
#     intensity = 0.6
#     X_resultant_kernel = cv2.getGaussianKernel(cols, cols * intensity)
#     Y_resultant_kernel = cv2.getGaussianKernel(rows, rows * intensity)
#     resultant_kernel = Y_resultant_kernel * X_resultant_kernel.T
#     mask = 255 * resultant_kernel / np.linalg.norm(resultant_kernel)
#     output = np.copy(input_image)
#     for i in range(3):
#         output[:,:,i] = output[:,:,i] * mask
#     return output

class VignetteEffect(A.ImageOnlyTransform):
    def __init__(self, intensity=0.6, always_apply=False, p=1.0):
        super().__init__(always_apply, p)
        self.intensity = intensity

    def apply(self, img, **params):
        rows, cols = img.shape[:2]
        X_resultant_kernel = cv2.getGaussianKernel(cols, cols * self.intensity)
        Y_resultant_kernel = cv2.getGaussianKernel(rows, rows * self.intensity)
        resultant_kernel = Y_resultant_kernel * X_resultant_kernel.T
        # mask = 255 * resultant_kernel / np.linalg.norm(resultant_kernel)
        mask = resultant_kernel / resultant_kernel.max()  # Normalize to range [0, 1]
        output = img.copy()
        for i in range(3):  # Loop over color channels
            output[:, :, i] = np.clip(output[:, :, i] * mask, 0, 255)
        return output.astype(np.uint8)


# ===================================

# def apply_frost(image, **kwargs):
#     alpha=0.5
#     frost_texture = cv2.imread("frost/4.jpg")
#     frost_texture = cv2.cvtColor(frost_texture, cv2.COLOR_BGR2RGB)
#     frost_texture = cv2.resize(frost_texture, (image.shape[1],image.shape[0]), interpolation=cv2.INTER_LINEAR_EXACT)
#     frosted_image = cv2.addWeighted(image, 1 - alpha, frost_texture, alpha, 0)
#     return frosted_image

class FrostEffect(A.ImageOnlyTransform):
    def __init__(self, frost_texture_path="frost/4.jpg", alpha=0.5, always_apply=False, p=1.0):
        super().__init__(always_apply, p)
        self.frost_texture_path = frost_texture_path
        self.alpha = alpha

    def apply(self, img, **params):
        frost_texture = cv2.imread(self.frost_texture_path)
        if frost_texture is None:
            raise ValueError(f"Frost texture image at path {self.frost_texture_path} could not be loaded.")

        frost_texture = cv2.cvtColor(frost_texture, cv2.COLOR_BGR2RGB)
        frost_texture = cv2.resize(frost_texture, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_LINEAR)
        
        if frost_texture.dtype != img.dtype:
            frost_texture = frost_texture.astype(img.dtype)
        
        frosted_image = cv2.addWeighted(img, 1 - self.alpha, frost_texture, self.alpha, 0)
        frosted_image = cv2.cvtColor(frosted_image, cv2.COLOR_RGB2BGR)
        return frosted_image

In [4]:
# l=glob("rasm/siamese/fold1.0/train/*/*")
# random.shuffle(l)

# Original=cv2.imread(l[0], cv2.IMREAD_UNCHANGED)
# Original=cv2.cvtColor(Original, cv2.COLOR_BGR2RGB)

# # 1.0. AutoContrast
# autocontrast_trans=A.Compose([AutoContrast(cutoff=3, p=1.0.0)], p=1.0.0)
# AutoContrast_img=autocontrast_trans(image=Original)["image"]

# # 2. Blur:
# Blur=A.Compose([A.Blur(p=1.0.0)], p=1.0.0)
# BoxBlur_img=Blur(image=Original)["image"]

# # 3. CLAHE
# CLAHE=A.Compose([A.CLAHE(clip_limit=2.0, p=1.0.0)], p=1.0.0)
# CLAHE_img=CLAHE(image=Original)['image']

# # 4. Channel Shuffling: Randomly rearranging RGB channels.
# ChannelShuffle=A.Compose([A.ChannelShuffle(p=1.0.0)], p=1.0.0)
# ChannelShuffle_img=ChannelShuffle(image=Original)["image"]

# # 5. ChromaticAberration
# ChromaticAberration=A.Compose([A.ChromaticAberration(mode='random',p=1.0.0)], p=1.0.0)
# ChromaticAberration_img=ChromaticAberration(image=Original)["image"]

# # 6. CoarseDropout: Masking random square regions.
# CoarseDropout=A.Compose([A.CoarseDropout(num_holes_range=(20,40),hole_height_range=(20,40),
#                                          hole_width_range=(20,40),p=1.0.0)],p=1.0.0)  # fill="inpaint_ns",
# CoarseDropout_img=CoarseDropout(image=Original)["image"]

# # 7. Color Jittering: Randomly varying brightness, contrast, saturation, and hue.
# ColorJitter=A.Compose([A.ColorJitter(brightness=(0.8,1.0.2),contrast=(0.8,1.0.2),saturation=(0.8,1.0.2),hue=(-0.5,0.5),p=1.0.0)],p=1.0.0)
# ColorJitter_img=ColorJitter(image=Original)['image'] 

# # 8. Defocus Blur:
# Defocus=A.Compose([A.Defocus(radius=(3,5),p=1.0.0)], p=1.0.0)
# Defocus_img=Defocus(image=Original)["image"]

# # 9. Dilation
# dilation=A.Compose([A.Morphological(scale=(3, 5), operation='dilation', p=1.0.0)], p=1.0.0)
# Dilation_img=dilation(image=Original)["image"]

# # 1.00. Elastic Transformations: Stretching or compressing areas in an elastic manner.
# ElasticTransform=A.Compose([A.ElasticTransform(p=1.0.0)],p=1.0.0)
# ElasticTransform_img=ElasticTransform(image=Original)['image']

# # 1.01.0. Emboss
# emboss=A.Compose([EmbossEffect(emboss_strength=1.0.0,p=1.0.0.0)],p=1.0.0)
# Emboss_img=emboss(image=Original)["image"]

# # 1.02. Histogram Equalization: Enhancing contrast by equalizing the histogram.
# Equalize=A.Compose([A.Equalize(p=1.0.0)], p=1.0.0)
# Equalize_img=Equalize(image=Original)['image']

# # 1.03. Histogram Equalization YUV: Enhancing contrast by equalizing the histogram.
# equalizeyuv_trans=A.Compose([EqualizeYUV(p=1.0.0)], p=1.0.0)
# EqualizeYUV_img=equalizeyuv_trans(image=Original)["image"]

# # 1.04. Erosion
# erosion=A.Compose([A.Morphological(scale=(3, 5), operation='erosion', p=1.0.0)], p=1.0.0)
# Erosion_img=erosion(image=Original)["image"]

# # 1.05. Frost
# frost=A.Compose([FrostEffect(frost_texture_path="frost/4.jpg", alpha=0.5, p=1.0.0)],p=1.0.0)
# Frost_img=frost(image=Original)["image"]

# # 1.06. Gamma Correction: 
# Gamma=A.Compose([A.RandomGamma(p=1.0.0)], p=1.0.0)
# Gamma_img=Gamma(image=Original)["image"] 

# # 1.07. Gaussian Noise: Adding random pixel values following a Gaussian distribution.
# GaussNoise=A.Compose([A.GaussNoise(p=1.0.0)], p=1.0.0)
# GaussNoise_img=GaussNoise(image=Original)["image"]

# # 1.08. Gaussian Blur: Applying a Gaussian kernel to blur the image.
# GaussianBlur=A.Compose([A.GaussianBlur(p=1.0.0)], p=1.0.0)
# GaussianBlur_img=GaussianBlur(image=Original)["image"]

# # 1.09. Glass Blur:
# GlassBlur=A.Compose([A.GlassBlur(p=1.0.0)], p=1.0.0)
# GlassBlur_img=GlassBlur(image=Original)["image"]

# # 20. Horizontal Flipping:  
# HorizontalFlip=A.Compose([A.HorizontalFlip(p=1.0.0)], p=1.0.0)
# HorizontalFlip_img=HorizontalFlip(image=Original)['image']

# # 21.0. Saturation Adjustment: Modifying the saturation of colors. Hue Adjustment: Changing the hue to alter color balance.
# HueSaturationValue=A.Compose([A.HueSaturationValue(p=1.0.0)], p=1.0.0)
# HueSaturationValue_img=HueSaturationValue(image=Original)["image"]  

# # 22. Image Invert:
# InvertImg=A.Compose([A.InvertImg(p=1.0.0)], p=1.0.0)
# Invert_img=InvertImg(image=Original)['image']

# # 23. JPEG Compression Artifacts: Simulating compression-induced noise.
# ImageCompression=A.Compose([A.ImageCompression(quality_lower=1.00,quality_upper=30,p=1.0.0)],p=1.0.0)
# JPEGCompression_img=ImageCompression(image=Original)["image"]

# # 24. Median Blur:
# MedianBlur=A.Compose([A.MedianBlur(p=1.0.0)], p=1.0.0)
# MedianBlur_img=MedianBlur(image=Original)["image"]

# # 25. Motion Blur: Simulating motion-induced blurring.   
# MotionBlur=A.Compose([A.MotionBlur(p=1.0.0)], p=1.0.0)
# MotionBlur_img=MotionBlur(image=Original)["image"]

# # 26. Optical Distortion
# OpticalDistortion=A.Compose([A.OpticalDistortion(distort_limit=0.5,p=1.0.0)],p=1.0.0)
# OpticalDistortion_img=OpticalDistortion(image=Original)['image']

# # 27. Original
# orig=A.Compose([A.NoOp(p=1.0.0)], p=1.0.0)
# Original_img=orig(image=Original)['image']

# # 28. Perspective Transformations: Simulating changes in camera angle.
# Perspective=A.Compose([A.Perspective(scale=0.2,p=1.0.0)],p=1.0.0)
# Perspective_img=Perspective(image=Original)['image']

# # 29. Posterize
# Posterize=A.Compose([A.Posterize(p=1.0.0)], p=1.0.0)
# Posterize_img=Posterize(image=Original)['image']

# # 30. RGBShift
# RGBShift=A.Compose([A.RGBShift(p=1.0.0)], p=1.0.0)
# RGBShift_img=RGBShift(image=Original)["image"]

# # 31.0. Brightness Adjustment: Increasing or decreasing brightness, Contrast Adjustment: Enhancing or reducing the contrast.
# RandomBrightnessContrast=A.Compose([A.RandomBrightnessContrast(p=1.0.0)], p=1.0.0)
# RandomBrightnessContrast_img=RandomBrightnessContrast(image=Original)['image']

# # 32. Adding fog 
# RandomFog=A.Compose([A.RandomFog(fog_coef_lower=0.5,fog_coef_upper=0.8,alpha_coef=0.2,p=1.0.0)],p=1.0.0)
# RandomFog_img=RandomFog(image=Original)["image"]

# # 33. Adding gravel 
# RandomGravel=A.Compose([A.RandomGravel(gravel_roi=(0,0,1.0,1.0),number_of_patches=40,p=1.0.0)],p=1.0.0)
# RandomGravel_img=RandomGravel(image=Original)["image"]

# # 34. RandomGridShuffle: Rearranging patches within an image.
# RandomGridShuffle=A.Compose([A.RandomGridShuffle(p=1.0.0)], p=1.0.0)
# RandomGridShuffle_img=RandomGridShuffle(image=Original)["image"]

# # 35. Adding rain 
# RandomRain=A.Compose([A.RandomRain(brightness_coefficient=0.9,drop_width=1.0,blur_value=3,p=1.0.0)],p=1.0.0)
# RandomRain_img=RandomRain(image=Original)["image"]

# # 36. Shadowing: Simulating shadows on objects.
# RandomShadow=A.Compose([A.RandomShadow(num_shadows_lower=2, num_shadows_upper=3, shadow_dimension=5, shadow_roi=(0,0,1.0,1.0), p=1.0.0)], p=1.0.0)
# RandomShadow_img=RandomShadow(image=Original)["image"]

# # 37. Adding snow 
# RandomSnow=A.Compose([A.RandomSnow(brightness_coeff=2, snow_point_lower=0.4, snow_point_upper=0.7, p=1.0.0)], p=1.0.0)
# RandomSnow_img=RandomSnow(image=Original)["image"]

# # 38. Adding sun flare 
# RandomSunFlare=A.Compose([A.RandomSunFlare(flare_roi=(0,0,1.0,1.0), src_radius=1.050, angle_lower=0.8, p=1.0.0)], p=1.0.0)
# RandomSunFlare_img=RandomSunFlare(image=Original)["image"]    

# # 39. Adding tone curve 
# RandomToneCurve=A.Compose([A.RandomToneCurve(scale=0.2, p=1.0.0)], p=1.0.0)
# RandomToneCurve_img=RandomToneCurve(image=Original)["image"]

# # 40. Rotation: Rotating the image by a specific angle.
# Rotate=A.Compose([A.augmentations.geometric.transforms.Affine(rotate=(-45,45), mode=2, p=1.0.0)], p=1.0.0)
# Rotate_img=Rotate(image=Original)['image']

# # 41.0. Salt-and-Pepper Noise: Randomly setting pixels to either black or white.
# salt_and_pepper=A.Compose([SaltAndPepperNoise(salt_prob=0.01.0, pepper_prob=0.01.0, p=1.0.0.0)], p=1.0.0)
# SaltandPepper_img=salt_and_pepper(image=Original)["image"]

# # 42. Scaling/Resizing: Zooming in or out on the image.
# Scale=A.Compose([A.augmentations.geometric.transforms.Affine(scale=(0.7,1.0.3), mode=2, p=1.0.0)], p=1.0.0)
# Scale_img=Scale(image=Original)['image']

# # 43. Sharpening: Enhancing edge details.
# Sharpen=A.Compose([A.Sharpen(p=1.0.0)], p=1.0.0)    
# Sharpen_img=Sharpen(image=Original)['image']

# # 44. ShearX: Applying a shear mapping to the image.
# ShearX=A.Compose([A.augmentations.geometric.transforms.Affine(shear = {'x': (-20,20)}, mode=2, p=1.0.0)], p=1.0.0)
# ShearX_img=ShearX(image=Original)['image']

# # 45. ShearY:
# ShearY=A.Compose([A.augmentations.geometric.transforms.Affine(shear = {'y': (-20,20)}, mode=2, p=1.0.0)], p=1.0.0)
# ShearY_img=ShearY(image=Original)['image']

# # 46. ShiftScaleRotate
# ShiftScaleRotate=A.Compose([A.ShiftScaleRotate(p=1.0.0)], p=1.0.0)
# ShiftScaleRotate_img=ShiftScaleRotate(image=Original)['image']

# # 47. Solarize
# Solarize=A.Compose([A.Solarize(p=1.0.0)], p=1.0.0)
# Solarize_img=Solarize(image=Original)['image']

# # 48. Speckle Noise: Multiplicative noise added to pixel intensities.
# speckle_noise=A.Compose([SpeckleNoise(noise_prob=0.1.0, p=1.0.0.0)], p=1.0.0.0)
# Speckle_img=speckle_noise(image=Original)["image"]

# # 49. ToGray
# ToGray=A.Compose([A.ToGray(method='weighted_average', p=1.0.0)], p=1.0.0) # weighted_average,from_lab,desaturation,average,max,pca
# ToGray_img=ToGray(image=Original)["image"]

# # 50. ToSepia
# ToSepia=A.Compose([A.ToSepia(p=1.0.0)], p=1.0.0)
# ToSepia_img=ToSepia(image=Original)["image"]

# # 51.0. Translation: Shifting the image along the x or y axis.
# Translate=A.Compose([A.augmentations.geometric.transforms.Affine(translate_percent=(-0.1.0,0.1.0), mode=2, p=1.0.0)], p=1.0.0)
# Translate_img=Translate(image=Original)['image']

# # 52. Vertical Flipping:
# VerticalFlip=A.Compose([A.VerticalFlip(p=1.0.0)], p=1.0.0)         
# VerticalFlip_img=VerticalFlip(image=Original)['image']

# # 53. Vignetting: Adding a vignette effect to the image.
# vignette_trans=A.Compose([VignetteEffect(intensity=0.6, p=1.0.0)], p=1.0.0)
# Vignette_img=vignette_trans(image=Original)["image"]

# # 54. Zoom Blur:
# ZoomBlur=A.Compose([A.ZoomBlur(p=1.0.0)], p=1.0.0)
# ZoomBlur_img=ZoomBlur(image=Original)["image"]

In [5]:
# transform_list = ['AutoContrast', 'BoxBlur', 'CLAHE', 'ChannelShuffle', 'ChromaticAberration', 'CoarseDropout', 
#                   'ColorJitter', 'Defocus', 'Dilation', 'ElasticTransform', 'Emboss', 'Equalize', 'EqualizeYUV', 
#                   'Erosion', 'Frost', 'Gamma', 'GaussNoise', 'GaussianBlur', 'GlassBlur', 'HorizontalFlip', 
#                   'HueSaturationValue', 'Invert', 'JPEGCompression', 'MedianBlur', 'MotionBlur', 'OpticalDistortion', 
#                   'Original', 'Perspective', 'Posterize', 'RGBShift', 'RandomBrightnessContrast', 'RandomFog', 
#                   'RandomGravel', 'RandomGridShuffle', 'RandomRain', 'RandomShadow', 'RandomSnow', 'RandomSunFlare', 
#                   'RandomToneCurve', 'Rotate', 'SaltandPepper', 'Scale', 'Sharpen', 'ShearX', 'ShearY', 'ShiftScaleRotate', 
#                   'Solarize', 'Speckle', 'ToGray', 'ToSepia', 'Translate', 'VerticalFlip', 'Vignette', 'ZoomBlur']

In [6]:
# print(len(transform_list))

In [7]:
# titles=['Original', 'AutoContrast', 'BoxBlur', 'CLAHE', 'ChannelShuffle', 'ChromaticAberration', 'CoarseDropout', 'ColorJitter', 
#         'Defocus', 'Dilation', 'ElasticTransform', 'Emboss', 'Equalize', 'EqualizeYUV', 'Erosion', 'Frost', 'Gamma', 
#         'GaussNoise', 'GaussianBlur', 'GlassBlur', 'HorizontalFlip', 'HueSaturationValue', 'Invert', 'JPEGCompression', 
#         'MedianBlur', 'MotionBlur', 'OpticalDistortion', 'Perspective', 'Posterize', 'RGBShift', 'RandomBrightnessContrast', 
#         'RandomFog', 'RandomGravel', 'RandomGridShuffle', 'RandomRain', 'RandomShadow', 'RandomSnow', 'RandomSunFlare', 
#         'RandomToneCurve', 'Rotate', 'SaltandPepper', 'Scale', 'Sharpen', 'ShearX', 'ShearY', 'ShiftScaleRotate', 'Solarize', 
#         'Speckle', 'ToGray', 'ToSepia', 'Translate', 'VerticalFlip', 'Vignette', 'ZoomBlur']

# images=[Original, AutoContrast_img, BoxBlur_img, CLAHE_img, ChannelShuffle_img, ChromaticAberration_img, CoarseDropout_img, ColorJitter_img,
#         Defocus_img, Dilation_img, ElasticTransform_img, Emboss_img, Equalize_img, EqualizeYUV_img, Erosion_img, Frost_img, Gamma_img,
#         GaussNoise_img, GaussianBlur_img, GlassBlur_img, HorizontalFlip_img, HueSaturationValue_img, Invert_img, JPEGCompression_img,
#         MedianBlur_img, MotionBlur_img, OpticalDistortion_img, Perspective_img, Posterize_img, RGBShift_img, RandomBrightnessContrast_img,
#         RandomFog_img, RandomGravel_img, RandomGridShuffle_img, RandomRain_img, RandomShadow_img, RandomSnow_img, RandomSunFlare_img,
#         RandomToneCurve_img, Rotate_img, SaltandPepper_img, Scale_img, Sharpen_img, ShearX_img, ShearY_img, ShiftScaleRotate_img,
#         Solarize_img, Speckle_img, ToGray_img, ToSepia_img, Translate_img, VerticalFlip_img, Vignette_img, ZoomBlur_img]

In [8]:
# fig=plt.figure(figsize=(1.04,1.08))
# fig.subplots_adjust(wspace=0.01.0, hspace=0.01.0)

# for i in range(len(images)):
#     plt.subplot(9,6,i+1.0)
#     plt.imshow(images[i], cmap='gray')
#     plt.title(titles[i])
#     plt.axis('off')

# plt.tight_layout()  
# plt.show()

In [9]:
# print('Total_Augmented_Images: ', len(images))

In [11]:
path="rasm/siamese/fold1/test/"
dest="rasm/siamese/fold1/test_aug/"

folders=sorted(os.listdir(path))
for i in folders:   
    os.mkdir(dest+i)
    files=os.listdir(path+i)   
    for x in files:  
        image_name=x.split('.')[0]   
        Original=cv2.imread(path+i+'/'+x)
        
        destination=dest+i+'/'+image_name+'/' 
        os.mkdir(destination)

        # l=glob("rasm/siamese/fold1/train/*/*")
        # random.shuffle(l)
        
        # Original=cv2.imread(l[0], cv2.IMREAD_UNCHANGED)
        # Original=cv2.cvtColor(Original, cv2.COLOR_BGR2RGB)
        
        # 1. AutoContrast
        autocontrast_trans=A.Compose([AutoContrast(cutoff=3, p=1.0)], p=1.0)
        AutoContrast_img=autocontrast_trans(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_AutoContrast.png", AutoContrast_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 2. Blur:
        Blur=A.Compose([A.Blur(p=1.0)], p=1.0)
        BoxBlur_img=Blur(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_BoxBlur.png", BoxBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 3. CLAHE
        CLAHE=A.Compose([A.CLAHE(clip_limit=2.0, p=1.0)], p=1.0)
        CLAHE_img=CLAHE(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_CLAHE.png", CLAHE_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 4. Channel Shuffling: Randomly rearranging RGB channels.
        ChannelShuffle=A.Compose([A.ChannelShuffle(p=1.0)], p=1.0)
        ChannelShuffle_img=ChannelShuffle(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ChannelShuffle.png", ChannelShuffle_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 5. ChromaticAberration
        ChromaticAberration=A.Compose([A.ChromaticAberration(mode='random',p=1.0)], p=1.0)
        ChromaticAberration_img=ChromaticAberration(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ChromaticAberration.png", ChromaticAberration_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 6. CoarseDropout: Masking random square regions.
        CoarseDropout=A.Compose([A.CoarseDropout(num_holes_range=(20,40),hole_height_range=(20,40),
                                                 hole_width_range=(20,40),p=1.0)],p=1.0)  # fill="inpaint_ns",
        CoarseDropout_img=CoarseDropout(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_CoarseDropout.png", CoarseDropout_img, [cv2.IMWRITE_PNG_COMPRESSION, 0]) 
        
        # 7. Color Jittering: Randomly varying brightness, contrast, saturation, and hue.
        ColorJitter=A.Compose([A.ColorJitter(brightness=(0.8,1.2),contrast=(0.8,1.2),saturation=(0.8,1.2),hue=(-0.5,0.5),p=1.0)],p=1.0)
        ColorJitter_img=ColorJitter(image=Original)['image'] 
        cv2.imwrite(destination+'{}'.format(image_name)+"_ColorJitter.png", ColorJitter_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 8. Defocus Blur:
        Defocus=A.Compose([A.Defocus(radius=(3,5),p=1.0)], p=1.0)
        Defocus_img=Defocus(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Defocus.png", Defocus_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 9. Dilation
        dilation=A.Compose([A.Morphological(scale=(3, 5), operation='dilation', p=1.0)], p=1.0)
        Dilation_img=dilation(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Dilation.png", Dilation_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 10. Elastic Transformations: Stretching or compressing areas in an elastic manner.
        ElasticTransform=A.Compose([A.ElasticTransform(p=1.0)],p=1.0)
        ElasticTransform_img=ElasticTransform(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_ElasticTransform.png", ElasticTransform_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 11. Emboss
        emboss=A.Compose([EmbossEffect(emboss_strength=1.0,p=1.0)],p=1.0)
        Emboss_img=emboss(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Emboss.png", Emboss_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 12. Histogram Equalization: Enhancing contrast by equalizing the histogram.
        Equalize=A.Compose([A.Equalize(p=1.0)], p=1.0)
        Equalize_img=Equalize(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Equalize.png", Equalize_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 13. Histogram Equalization YUV: Enhancing contrast by equalizing the histogram.
        equalizeyuv_trans=A.Compose([EqualizeYUV(p=1.0)], p=1.0)
        EqualizeYUV_img=equalizeyuv_trans(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_EqualizeYUV.png", EqualizeYUV_img, [cv2.IMWRITE_PNG_COMPRESSION, 0]) 
        
        # 14. Erosion
        erosion=A.Compose([A.Morphological(scale=(3, 5), operation='erosion', p=1.0)], p=1.0)
        Erosion_img=erosion(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Erosion.png", Erosion_img, [cv2.IMWRITE_PNG_COMPRESSION, 0]) 
        
        # 15. Frost
        frost=A.Compose([FrostEffect(frost_texture_path="frost/4.jpg", alpha=0.5, p=1.0)],p=1.0)
        Frost_img=frost(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Frost.png", Frost_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 16. Gamma Correction: 
        Gamma=A.Compose([A.RandomGamma(p=1.0)], p=1.0)
        Gamma_img=Gamma(image=Original)["image"] 
        cv2.imwrite(destination+'{}'.format(image_name)+"_Gamma.png", Gamma_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 17. Gaussian Noise: Adding random pixel values following a Gaussian distribution.
        GaussNoise=A.Compose([A.GaussNoise(p=1.0)], p=1.0)
        GaussNoise_img=GaussNoise(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_GaussNoise.png", GaussNoise_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 18. Gaussian Blur: Applying a Gaussian kernel to blur the image.
        GaussianBlur=A.Compose([A.GaussianBlur(p=1.0)], p=1.0)
        GaussianBlur_img=GaussianBlur(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_GaussianBlur.png", GaussianBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 19. Glass Blur:
        GlassBlur=A.Compose([A.GlassBlur(p=1.0)], p=1.0)
        GlassBlur_img=GlassBlur(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_GlassBlur.png", GlassBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0]) 
        
        # 20. Horizontal Flipping:  
        HorizontalFlip=A.Compose([A.HorizontalFlip(p=1.0)], p=1.0)
        HorizontalFlip_img=HorizontalFlip(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_HorizontalFlip.png", HorizontalFlip_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 21. Saturation Adjustment: Modifying the saturation of colors. Hue Adjustment: Changing the hue to alter color balance.
        HueSaturationValue=A.Compose([A.HueSaturationValue(p=1.0)], p=1.0)
        HueSaturationValue_img=HueSaturationValue(image=Original)["image"]  
        cv2.imwrite(destination+'{}'.format(image_name)+"_HueSaturationValue.png", HueSaturationValue_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 22. Image Invert:
        InvertImg=A.Compose([A.InvertImg(p=1.0)], p=1.0)
        Invert_img=InvertImg(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Invert.png", Invert_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 23. JPEG Compression Artifacts: Simulating compression-induced noise.
        ImageCompression=A.Compose([A.ImageCompression(quality_lower=10,quality_upper=30,p=1.0)],p=1.0)
        JPEGCompression_img=ImageCompression(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_JPEGCompression.png", JPEGCompression_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 24. Median Blur:
        MedianBlur=A.Compose([A.MedianBlur(p=1.0)], p=1.0)
        MedianBlur_img=MedianBlur(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_MedianBlur.png", MedianBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 25. Motion Blur: Simulating motion-induced blurring.   
        MotionBlur=A.Compose([A.MotionBlur(p=1.0)], p=1.0)
        MotionBlur_img=MotionBlur(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_MotionBlur.png", MotionBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 26. Optical Distortion
        OpticalDistortion=A.Compose([A.OpticalDistortion(distort_limit=0.5,p=1.0)],p=1.0)
        OpticalDistortion_img=OpticalDistortion(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_OpticalDistortion.png", OpticalDistortion_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 27. Original
        orig=A.Compose([A.NoOp(p=1.0)], p=1.0)
        Original_img=orig(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Original.png", Original_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 28. Perspective Transformations: Simulating changes in camera angle.
        Perspective=A.Compose([A.Perspective(scale=0.2,p=1.0)],p=1.0)
        Perspective_img=Perspective(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Perspective.png", Perspective_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 29. Posterize
        Posterize=A.Compose([A.Posterize(p=1.0)], p=1.0)
        Posterize_img=Posterize(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Posterize.png", Posterize_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
                
        # 30. RGBShift
        RGBShift=A.Compose([A.RGBShift(p=1.0)], p=1.0)
        RGBShift_img=RGBShift(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RGBShift.png", RGBShift_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])        
        
        # 31. Brightness Adjustment: Increasing or decreasing brightness, Contrast Adjustment: Enhancing or reducing the contrast.
        RandomBrightnessContrast=A.Compose([A.RandomBrightnessContrast(p=1.0)], p=1.0)
        RandomBrightnessContrast_img=RandomBrightnessContrast(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomBrightnessContrast.png", RandomBrightnessContrast_img, [cv2.IMWRITE_PNG_COMPRESSION, 0]) 
        
        # 32. Adding fog 
        RandomFog=A.Compose([A.RandomFog(fog_coef_lower=0.5,fog_coef_upper=0.8,alpha_coef=0.2,p=1.0)],p=1.0)
        RandomFog_img=RandomFog(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomFog.png", RandomFog_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 33. Adding gravel 
        RandomGravel=A.Compose([A.RandomGravel(gravel_roi=(0,0,1,1),number_of_patches=40,p=1.0)],p=1.0)
        RandomGravel_img=RandomGravel(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomGravel.png", RandomGravel_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 34. RandomGridShuffle: Rearranging patches within an image.
        RandomGridShuffle=A.Compose([A.RandomGridShuffle(p=1.0)], p=1.0)
        RandomGridShuffle_img=RandomGridShuffle(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomGridShuffle.png", RandomGridShuffle_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 35. Adding rain 
        RandomRain=A.Compose([A.RandomRain(brightness_coefficient=0.9,drop_width=1,blur_value=3,p=1.0)],p=1.0)
        RandomRain_img=RandomRain(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomRain.png", RandomRain_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 36. Shadowing: Simulating shadows on objects.
        RandomShadow=A.Compose([A.RandomShadow(num_shadows_lower=2, num_shadows_upper=3, shadow_dimension=5, shadow_roi=(0,0,1,1), p=1.0)], p=1.0)
        RandomShadow_img=RandomShadow(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomShadow.png", RandomShadow_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 37. Adding snow 
        RandomSnow=A.Compose([A.RandomSnow(brightness_coeff=2, snow_point_lower=0.4, snow_point_upper=0.7, p=1.0)], p=1.0)
        RandomSnow_img=RandomSnow(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomSnow.png", RandomSnow_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])  
        
        # 38. Adding sun flare 
        RandomSunFlare=A.Compose([A.RandomSunFlare(flare_roi=(0,0,1,1), src_radius=150, angle_lower=0.8, p=1.0)], p=1.0)
        RandomSunFlare_img=RandomSunFlare(image=Original)["image"]   
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomSunFlare.png", RandomSunFlare_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 39. Adding tone curve 
        RandomToneCurve=A.Compose([A.RandomToneCurve(scale=0.2, p=1.0)], p=1.0)
        RandomToneCurve_img=RandomToneCurve(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomToneCurve.png", RandomToneCurve_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 40. Rotation: Rotating the image by a specific angle.
        Rotate=A.Compose([A.augmentations.geometric.transforms.Affine(rotate=(-45,45), mode=2, p=1.0)], p=1.0)
        Rotate_img=Rotate(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Rotate.png", Rotate_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 41. Salt-and-Pepper Noise: Randomly setting pixels to either black or white.
        salt_and_pepper=A.Compose([SaltAndPepperNoise(salt_prob=0.01, pepper_prob=0.01, p=1.0)], p=1.0)
        SaltandPepper_img=salt_and_pepper(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_SaltandPepper.png", SaltandPepper_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 42. Scaling/Resizing: Zooming in or out on the image.
        Scale=A.Compose([A.augmentations.geometric.transforms.Affine(scale=(0.7,1.3), mode=2, p=1.0)], p=1.0)
        Scale_img=Scale(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Scale.png", Scale_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 43. Sharpening: Enhancing edge details.
        Sharpen=A.Compose([A.Sharpen(p=1.0)], p=1.0)    
        Sharpen_img=Sharpen(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Sharpen.png", Sharpen_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 44. ShearX: Applying a shear mapping to the image.
        ShearX=A.Compose([A.augmentations.geometric.transforms.Affine(shear = {'x': (-20,20)}, mode=2, p=1.0)], p=1.0)
        ShearX_img=ShearX(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_ShearX.png", ShearX_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 45. ShearY:
        ShearY=A.Compose([A.augmentations.geometric.transforms.Affine(shear = {'y': (-20,20)}, mode=2, p=1.0)], p=1.0)
        ShearY_img=ShearY(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_ShearY.png", ShearY_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 46. ShiftScaleRotate
        ShiftScaleRotate=A.Compose([A.ShiftScaleRotate(p=1.0)], p=1.0)
        ShiftScaleRotate_img=ShiftScaleRotate(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_ShiftScaleRotate.png", ShiftScaleRotate_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 47. Solarize
        Solarize=A.Compose([A.Solarize(p=1.0)], p=1.0)
        Solarize_img=Solarize(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Solarize.png", Solarize_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
                
        # 48. Speckle Noise: Multiplicative noise added to pixel intensities.
        speckle_noise=A.Compose([SpeckleNoise(noise_prob=0.1, p=1.0)], p=1.0)
        Speckle_img=speckle_noise(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Speckle.png", Speckle_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 49. ToGray
        ToGray=A.Compose([A.ToGray(method='weighted_average', p=1.0)], p=1.0) # weighted_average,from_lab,desaturation,average,max,pca
        ToGray_img=ToGray(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ToGray.png", ToGray_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 50. ToSepia
        ToSepia=A.Compose([A.ToSepia(p=1.0)], p=1.0)
        ToSepia_img=ToSepia(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ToSepia.png", ToSepia_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 51. Translation: Shifting the image along the x or y axis.
        Translate=A.Compose([A.augmentations.geometric.transforms.Affine(translate_percent=(-0.1,0.1), mode=2, p=1.0)], p=1.0)
        Translate_img=Translate(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Translate.png", Translate_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 52. Vertical Flipping:
        VerticalFlip=A.Compose([A.VerticalFlip(p=1.0)], p=1.0)         
        VerticalFlip_img=VerticalFlip(image=Original)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_VerticalFlip.png", VerticalFlip_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 53. Vignetting: Adding a vignette effect to the image.
        vignette_trans=A.Compose([VignetteEffect(intensity=0.6, p=1.0)], p=1.0)
        Vignette_img=vignette_trans(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Vignette.png", Vignette_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 54. Zoom Blur:
        ZoomBlur=A.Compose([A.ZoomBlur(p=1.0)], p=1.0)
        ZoomBlur_img=ZoomBlur(image=Original)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ZoomBlur.png", ZoomBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])        